# Module 8 • Large Language Models

# Lesson 46 • Advanced RAG — Dense Embeddings, Re-Ranking, and Retrieval Evaluation

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Advanced  
**Estimated study time:** 180–220 minutes  
**Execution target:** CPU only

---

## Scope

This lesson extends the basic RAG pipeline with stronger retrieval techniques.

The executable core is fully offline and demonstrates:

- sparse TF-IDF retrieval;
- dense latent-semantic retrieval with truncated SVD;
- hybrid sparse+dense scoring;
- metadata filters;
- query expansion;
- two-stage retrieval;
- lexical-semantic reranking;
- recall@k, precision@k, MRR, MAP, and nDCG;
- hard-negative analysis;
- chunk-level versus document-level evaluation;
- context selection under a token/word budget;
- retrieval ablations.

Optional cells explain how to replace the offline dense representation with a
neural sentence-embedding model.

## Learning Objectives

After completing this lesson, the learner should be able to:

- distinguish sparse and dense retrieval;
- explain how latent semantic projections approximate dense retrieval;
- combine sparse and dense relevance signals;
- apply metadata filters before or after vector search;
- perform query expansion;
- construct a two-stage retriever;
- rerank candidate passages;
- compute MAP and nDCG in addition to recall@k and MRR;
- analyze hard negatives;
- compare document-level and chunk-level evaluation;
- enforce context budgets;
- run retrieval ablations;
- identify multilingual and Arabic dense-retrieval challenges.

## Table of Contents

1. Why Advanced Retrieval?
2. Sparse Retrieval
3. Dense Retrieval
4. Latent Semantic Embeddings
5. Hybrid Retrieval
6. Metadata Filtering
7. Query Expansion
8. Two-Stage Retrieval
9. Re-Ranking
10. Cross-Encoder Concept
11. Hard Negatives
12. Offline Knowledge Base
13. Chunking
14. Sparse Index
15. Dense SVD Index
16. Sparse Search
17. Dense Search
18. Hybrid Search
19. Metadata-Aware Search
20. Query Expansion Search
21. Candidate Generation
22. Heuristic Re-Ranker
23. Two-Stage Retrieval Pipeline
24. Retrieval Benchmark
25. Recall@k
26. Precision@k
27. Reciprocal Rank
28. Average Precision
29. Mean Average Precision
30. nDCG
31. Retriever Comparison
32. Chunk-Level Evaluation
33. Document-Level Evaluation
34. Hard-Negative Analysis
35. Context Budgeting
36. Diversity-Aware Context Selection
37. Retrieval Ablation
38. Failure Taxonomy
39. Dense Embedding Model Considerations
40. Vector Database Considerations
41. Multilingual Retrieval
42. Arabic Dense Retrieval
43. Optional Sentence-Transformer Workflow
44. Reproducibility
45. Knowledge Check
46. Exercises
47. Summary and Next Lesson

# 1. Why Advanced Retrieval?

Basic top-k similarity search is often insufficient because:

- lexical overlap can miss paraphrases;
- dense retrieval can miss exact identifiers;
- top results can contain near duplicates;
- metadata constraints may matter;
- first-stage scores may not rank candidates optimally.

Advanced RAG therefore treats retrieval as a multi-stage ranking problem.

In [ ]:
import importlib.util
import math
import platform
import re
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import Normalizer

advanced_retrieval_layers = pd.DataFrame(
    [
        ("Candidate generation", "high recall"),
        ("Filtering", "respect metadata and permissions"),
        ("Re-ranking", "improve top-order precision"),
        ("Context selection", "fit evidence budget"),
    ],
    columns=["Stage", "Primary goal"],
)

advanced_retrieval_layers

# 2. Sparse Retrieval

Sparse retrieval represents text using high-dimensional lexical features.

Strengths:

- exact terms;
- product codes;
- names;
- technical identifiers;
- interpretability.

Weakness:

- lexical mismatch.

# 3. Dense Retrieval

Dense retrieval maps text into lower-dimensional continuous vectors.

Strength:

- semantic similarity and paraphrase matching.

Weaknesses:

- embedding model quality matters;
- exact rare identifiers may be underweighted;
- indexing and serving are more expensive.

# 4. Latent Semantic Embeddings

This notebook uses TF-IDF followed by truncated SVD as an offline approximation
of dense semantic embeddings.

It is not equivalent to a modern neural embedding model, but it lets us study
dense-vector retrieval mechanics without external downloads.

# 5. Hybrid Retrieval

Hybrid retrieval combines sparse and dense scores:

\[
s_{hybrid} = \lambda s_{sparse} + (1-\lambda)s_{dense}
\]

The mixing weight should be tuned on validation queries.

# 6. Metadata Filtering

Metadata can represent:

- language;
- source;
- date;
- department;
- document type;
- access-control group.

Filters can be applied before scoring or after candidate generation.

# 7. Query Expansion

Query expansion adds related terms to reduce vocabulary mismatch.

Sources of expansion terms can include:

- synonyms;
- user profile terms;
- entity aliases;
- acronym expansions;
- model-generated rewrites.

# 8. Two-Stage Retrieval

A common architecture is:

```text
query -> retrieve many candidates -> rerank candidates -> keep best few
```

# 9. Re-Ranking

Re-rankers use stronger relevance features on a small candidate pool.

# 10. Cross-Encoder Concept

A cross-encoder jointly processes query and candidate text and predicts a
relevance score.

It is often more accurate than independent embeddings but much slower, so it is
typically used only on a small candidate set.

# 11. Hard Negatives

Hard negatives are non-relevant passages that look deceptively similar to the
query.

They are useful for both training and evaluation because they expose ranking
weaknesses.

# 12. Offline Knowledge Base

In [ ]:
documents = [
    {
        "document_id": "doc_rag_architecture",
        "title": "RAG Architecture",
        "language": "en",
        "topic": "rag",
        "text": (
            "Retrieval-augmented generation separates retrieval from generation. "
            "The retriever selects evidence from an external knowledge base. "
            "The generator receives the user query and retrieved evidence. "
            "Grounded responses should remain supported by that evidence."
        ),
    },
    {
        "document_id": "doc_rag_metrics",
        "title": "Retrieval Metrics",
        "language": "en",
        "topic": "evaluation",
        "text": (
            "Recall at k measures whether relevant evidence appears in the first k results. "
            "Precision at k measures the relevant fraction of retrieved results. "
            "Mean reciprocal rank rewards placing the first relevant result near the top. "
            "Average precision rewards ranking multiple relevant results early."
        ),
    },
    {
        "document_id": "doc_ndcg",
        "title": "Ranked Retrieval Metrics",
        "language": "en",
        "topic": "evaluation",
        "text": (
            "Normalized discounted cumulative gain evaluates ranked retrieval with graded relevance. "
            "Higher-ranked relevant results receive more credit than lower-ranked results. "
            "nDCG is useful when relevance is not strictly binary."
        ),
    },
    {
        "document_id": "doc_sparse",
        "title": "Sparse Search",
        "language": "en",
        "topic": "retrieval",
        "text": (
            "Sparse retrieval uses lexical signals such as TF-IDF or BM25. "
            "It performs well for exact terminology, names, codes, and identifiers. "
            "Sparse methods can struggle when the query paraphrases the evidence."
        ),
    },
    {
        "document_id": "doc_dense",
        "title": "Dense Search",
        "language": "en",
        "topic": "retrieval",
        "text": (
            "Dense retrieval maps queries and passages into continuous embedding vectors. "
            "Semantic similarity can retrieve paraphrases even when exact words differ. "
            "Dense retrieval quality depends strongly on the embedding model and training domain."
        ),
    },
    {
        "document_id": "doc_hybrid",
        "title": "Hybrid Retrieval",
        "language": "en",
        "topic": "retrieval",
        "text": (
            "Hybrid retrieval combines lexical and dense semantic relevance signals. "
            "Sparse scores help preserve exact-term matching while dense scores improve paraphrase recall. "
            "The mixing weight should be tuned on validation data."
        ),
    },
    {
        "document_id": "doc_reranking",
        "title": "Re-Ranking",
        "language": "en",
        "topic": "retrieval",
        "text": (
            "A reranker applies a stronger relevance model to candidates from a first-stage retriever. "
            "Cross-encoders jointly process the query and candidate passage. "
            "Reranking is slower than vector search but can improve top-rank precision."
        ),
    },
    {
        "document_id": "doc_metadata",
        "title": "Metadata Filters",
        "language": "en",
        "topic": "systems",
        "text": (
            "Metadata filters restrict retrieval using attributes such as language, department, date, and access group. "
            "Filtering can prevent irrelevant or unauthorized content from entering the candidate set."
        ),
    },
    {
        "document_id": "doc_arabic",
        "title": "Arabic Retrieval",
        "language": "ar",
        "topic": "multilingual",
        "text": (
            "Arabic retrieval is affected by rich morphology, clitics, orthographic variation, and optional tashkeel. "
            "For fully vocalized tasks, tashkeel should be preserved consistently during indexing and querying. "
            "Evaluation should test both lexical and semantic matching under Arabic morphology."
        ),
    },
    {
        "document_id": "doc_context",
        "title": "Context Selection",
        "language": "en",
        "topic": "rag",
        "text": (
            "Retrieved passages must fit inside the generator context budget. "
            "Context selection should balance relevance, diversity, and evidence coverage. "
            "Near-duplicate chunks can waste context space."
        ),
    },
]

pd.DataFrame(documents)[
    ["document_id", "title", "language", "topic"]
]

# 13. Chunking

In [ ]:
SENTENCE_PATTERN = re.compile(
    r"(?<=[.!?])\s+"
)


def split_sentences(text: str) -> list[str]:
    return [
        sentence.strip()
        for sentence in SENTENCE_PATTERN.split(
            text.strip()
        )
        if sentence.strip()
    ]


def chunk_documents(
    documents,
    sentences_per_chunk=2,
    overlap_sentences=1,
):
    chunks = []
    stride = (
        sentences_per_chunk
        - overlap_sentences
    )

    for document in documents:
        sentences = split_sentences(
            document["text"]
        )

        chunk_number = 0

        for start in range(
            0,
            len(sentences),
            stride,
        ):
            selected = sentences[
                start:
                start + sentences_per_chunk
            ]

            if not selected:
                continue

            chunks.append(
                {
                    "chunk_id": (
                        f"{document['document_id']}"
                        f"_chunk_{chunk_number:02d}"
                    ),
                    "document_id": document[
                        "document_id"
                    ],
                    "title": document[
                        "title"
                    ],
                    "language": document[
                        "language"
                    ],
                    "topic": document[
                        "topic"
                    ],
                    "text": " ".join(
                        selected
                    ),
                }
            )

            chunk_number += 1

            if (
                start
                + sentences_per_chunk
                >= len(sentences)
            ):
                break

    return pd.DataFrame(
        chunks
    )


chunk_frame = chunk_documents(
    documents
)

chunk_frame.head()

# 14. Sparse Index

In [ ]:
sparse_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
)

sparse_matrix = sparse_vectorizer.fit_transform(
    chunk_frame["text"]
)

sparse_matrix.shape

# 15. Dense SVD Index

In [ ]:
maximum_components = min(
    16,
    sparse_matrix.shape[0] - 1,
    sparse_matrix.shape[1] - 1,
)

svd = TruncatedSVD(
    n_components=maximum_components,
    random_state=42,
)

dense_matrix = svd.fit_transform(
    sparse_matrix
)

normalizer = Normalizer()

dense_matrix = normalizer.fit_transform(
    dense_matrix
)

print(
    "Dense embedding shape:",
    dense_matrix.shape,
)

# 16. Sparse Search

In [ ]:
def sparse_scores(
    query: str,
) -> np.ndarray:
    query_vector = (
        sparse_vectorizer.transform(
            [query]
        )
    )

    return cosine_similarity(
        query_vector,
        sparse_matrix,
    )[0]


def rank_frame(
    scores: np.ndarray,
    top_k: int,
    mask: np.ndarray | None = None,
) -> pd.DataFrame:
    working_scores = scores.copy()

    if mask is not None:
        working_scores[
            ~mask
        ] = -np.inf

    indices = np.argsort(
        working_scores
    )[::-1]

    rows = []

    for index in indices:
        score = float(
            working_scores[index]
        )

        if not np.isfinite(score):
            continue

        row = chunk_frame.iloc[
            int(index)
        ]

        rows.append(
            {
                "rank": len(rows) + 1,
                "score": score,
                "chunk_id": row.chunk_id,
                "document_id": (
                    row.document_id
                ),
                "title": row.title,
                "language": row.language,
                "topic": row.topic,
                "text": row.text,
            }
        )

        if len(rows) >= top_k:
            break

    return pd.DataFrame(rows)


rank_frame(
    sparse_scores(
        "How does hybrid retrieval combine lexical and semantic matching?"
    ),
    top_k=5,
)

# 17. Dense Search

In [ ]:
def dense_scores(
    query: str,
) -> np.ndarray:
    sparse_query = (
        sparse_vectorizer.transform(
            [query]
        )
    )

    dense_query = svd.transform(
        sparse_query
    )

    dense_query = (
        normalizer.transform(
            dense_query
        )
    )

    return cosine_similarity(
        dense_query,
        dense_matrix,
    )[0]


rank_frame(
    dense_scores(
        "Which method finds paraphrases even without exact words?"
    ),
    top_k=5,
)

# 18. Hybrid Search

In [ ]:
def minmax_scale(
    values: np.ndarray,
) -> np.ndarray:
    values = np.asarray(
        values,
        dtype=float,
    )

    minimum = np.min(values)
    maximum = np.max(values)

    if (
        maximum
        - minimum
        < 1e-12
    ):
        return np.zeros_like(
            values
        )

    return (
        values
        - minimum
    ) / (
        maximum
        - minimum
    )


def hybrid_scores(
    query: str,
    sparse_weight: float = 0.5,
) -> np.ndarray:
    sparse = minmax_scale(
        sparse_scores(query)
    )

    dense = minmax_scale(
        dense_scores(query)
    )

    return (
        sparse_weight
        * sparse
        + (
            1.0
            - sparse_weight
        )
        * dense
    )


rank_frame(
    hybrid_scores(
        "How can exact identifiers and semantic paraphrases both be retrieved?",
        sparse_weight=0.55,
    ),
    top_k=5,
)

# 19. Metadata-Aware Search

In [ ]:
def metadata_mask(
    language: str | None = None,
    topic: str | None = None,
) -> np.ndarray:
    mask = np.ones(
        len(chunk_frame),
        dtype=bool,
    )

    if language is not None:
        mask &= (
            chunk_frame[
                "language"
            ].to_numpy()
            == language
        )

    if topic is not None:
        mask &= (
            chunk_frame[
                "topic"
            ].to_numpy()
            == topic
        )

    return mask


rank_frame(
    hybrid_scores(
        "What should Arabic retrieval preserve?"
    ),
    top_k=4,
    mask=metadata_mask(
        language="ar"
    ),
)

# 20. Query Expansion Search

In [ ]:
EXPANSIONS = {
    "rerank": [
        "reranker",
        "cross encoder",
        "candidate ranking",
    ],
    "metrics": [
        "recall",
        "precision",
        "mrr",
        "average precision",
        "ndcg",
    ],
    "arabic": [
        "morphology",
        "clitics",
        "tashkeel",
    ],
}


def expand_query(
    query: str,
) -> str:
    lowered = query.lower()

    additions = []

    for trigger, terms in (
        EXPANSIONS.items()
    ):
        if trigger in lowered:
            additions.extend(
                terms
            )

    if not additions:
        return query

    return (
        query
        + " "
        + " ".join(additions)
    )


original_query = (
    "Which metrics evaluate retrieval?"
)

expanded_query = expand_query(
    original_query
)

print(
    "Original:",
    original_query,
)
print(
    "Expanded:",
    expanded_query,
)

# 21. Candidate Generation

In [ ]:
def retrieve_candidates(
    query: str,
    candidate_k: int = 8,
    sparse_weight: float = 0.5,
    language: str | None = None,
    topic: str | None = None,
    use_expansion: bool = False,
) -> pd.DataFrame:
    working_query = (
        expand_query(query)
        if use_expansion
        else query
    )

    scores = hybrid_scores(
        working_query,
        sparse_weight=(
            sparse_weight
        ),
    )

    mask = metadata_mask(
        language=language,
        topic=topic,
    )

    return rank_frame(
        scores,
        top_k=candidate_k,
        mask=mask,
    )


retrieve_candidates(
    "How does reranking improve retrieval?",
    candidate_k=6,
    use_expansion=True,
)

# 22. Heuristic Re-Ranker

The offline reranker combines:

- query-token coverage;
- phrase overlap;
- first-stage relevance score.

It is a deterministic stand-in for a learned cross-encoder reranker.

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\b\w+\b",
    flags=re.UNICODE,
)


def text_tokens(
    text: str,
) -> list[str]:
    return TOKEN_PATTERN.findall(
        text.lower()
    )


def rerank_score(
    query: str,
    passage: str,
    first_stage_score: float,
) -> float:
    query_tokens = text_tokens(
        query
    )

    passage_tokens = text_tokens(
        passage
    )

    if not query_tokens:
        return float(
            first_stage_score
        )

    query_set = set(
        query_tokens
    )

    passage_set = set(
        passage_tokens
    )

    token_coverage = (
        len(
            query_set
            & passage_set
        )
        / len(query_set)
    )

    query_bigrams = set(
        zip(
            query_tokens,
            query_tokens[1:],
        )
    )

    passage_bigrams = set(
        zip(
            passage_tokens,
            passage_tokens[1:],
        )
    )

    bigram_overlap = (
        len(
            query_bigrams
            & passage_bigrams
        )
        / max(
            len(query_bigrams),
            1,
        )
    )

    return (
        0.55
        * first_stage_score
        + 0.30
        * token_coverage
        + 0.15
        * bigram_overlap
    )


def rerank_candidates(
    query: str,
    candidates: pd.DataFrame,
    top_k: int = 3,
) -> pd.DataFrame:
    reranked = candidates.copy()

    reranked[
        "rerank_score"
    ] = [
        rerank_score(
            query,
            row.text,
            row.score,
        )
        for row in reranked.itertuples(
            index=False
        )
    ]

    reranked = (
        reranked.sort_values(
            "rerank_score",
            ascending=False,
        )
        .head(top_k)
        .reset_index(
            drop=True
        )
    )

    reranked[
        "rank"
    ] = np.arange(
        1,
        len(reranked) + 1,
    )

    return reranked


candidates = retrieve_candidates(
    "Why is a cross encoder useful after vector retrieval?",
    candidate_k=8,
)

rerank_candidates(
    "Why is a cross encoder useful after vector retrieval?",
    candidates,
    top_k=4,
)

# 23. Two-Stage Retrieval Pipeline

In [ ]:
def advanced_retrieve(
    query: str,
    candidate_k: int = 8,
    top_k: int = 3,
    sparse_weight: float = 0.5,
    language: str | None = None,
    topic: str | None = None,
    use_expansion: bool = False,
) -> pd.DataFrame:
    candidates = (
        retrieve_candidates(
            query,
            candidate_k=(
                candidate_k
            ),
            sparse_weight=(
                sparse_weight
            ),
            language=language,
            topic=topic,
            use_expansion=(
                use_expansion
            ),
        )
    )

    return rerank_candidates(
        query,
        candidates,
        top_k=top_k,
    )


advanced_retrieve(
    "How should retrieval results be reranked?",
    candidate_k=8,
    top_k=3,
)

# 24. Retrieval Benchmark

In [ ]:
benchmark = [
    {
        "query": (
            "Which metric rewards putting the first relevant result near the top?"
        ),
        "relevant_documents": {
            "doc_rag_metrics"
        },
        "graded_relevance": {
            "doc_rag_metrics": 3,
            "doc_ndcg": 1,
        },
    },
    {
        "query": (
            "What retrieval method is good for paraphrases without exact lexical overlap?"
        ),
        "relevant_documents": {
            "doc_dense"
        },
        "graded_relevance": {
            "doc_dense": 3,
            "doc_hybrid": 2,
        },
    },
    {
        "query": (
            "How can exact terms and semantic similarity be combined?"
        ),
        "relevant_documents": {
            "doc_hybrid"
        },
        "graded_relevance": {
            "doc_hybrid": 3,
            "doc_sparse": 1,
            "doc_dense": 1,
        },
    },
    {
        "query": (
            "Why are cross encoders used after first stage retrieval?"
        ),
        "relevant_documents": {
            "doc_reranking"
        },
        "graded_relevance": {
            "doc_reranking": 3,
        },
    },
    {
        "query": (
            "What should fully vocalized Arabic retrieval preserve?"
        ),
        "relevant_documents": {
            "doc_arabic"
        },
        "graded_relevance": {
            "doc_arabic": 3,
        },
    },
    {
        "query": (
            "Why should retrieved context avoid duplicate passages?"
        ),
        "relevant_documents": {
            "doc_context"
        },
        "graded_relevance": {
            "doc_context": 3,
            "doc_rag_architecture": 1,
        },
    },
]

len(benchmark)

# 25. Recall@k

In [ ]:
def recall_at_k(
    ranked_documents,
    relevant_documents,
    k,
):
    selected = set(
        ranked_documents[:k]
    )

    return (
        len(
            selected
            & relevant_documents
        )
        / max(
            len(
                relevant_documents
            ),
            1,
        )
    )

# 26. Precision@k

In [ ]:
def precision_at_k(
    ranked_documents,
    relevant_documents,
    k,
):
    selected = (
        ranked_documents[:k]
    )

    if not selected:
        return 0.0

    return (
        sum(
            document_id
            in relevant_documents
            for document_id
            in selected
        )
        / len(selected)
    )

# 27. Reciprocal Rank

In [ ]:
def reciprocal_rank(
    ranked_documents,
    relevant_documents,
):
    for rank, document_id in enumerate(
        ranked_documents,
        start=1,
    ):
        if (
            document_id
            in relevant_documents
        ):
            return 1.0 / rank

    return 0.0

# 28. Average Precision

In [ ]:
def average_precision(
    ranked_documents,
    relevant_documents,
):
    if not relevant_documents:
        return 0.0

    precision_values = []
    relevant_seen = 0

    for rank, document_id in enumerate(
        ranked_documents,
        start=1,
    ):
        if (
            document_id
            in relevant_documents
        ):
            relevant_seen += 1

            precision_values.append(
                relevant_seen
                / rank
            )

    if not precision_values:
        return 0.0

    return (
        sum(
            precision_values
        )
        / len(
            relevant_documents
        )
    )

# 29. Mean Average Precision

MAP is the mean of average precision across benchmark queries.

# 30. nDCG

In [ ]:
def dcg(
    relevance_values,
):
    return sum(
        (
            (2 ** relevance - 1)
            / math.log2(
                rank + 1
            )
        )
        for rank, relevance
        in enumerate(
            relevance_values,
            start=1,
        )
    )


def ndcg(
    ranked_documents,
    graded_relevance,
    k,
):
    gains = [
        graded_relevance.get(
            document_id,
            0,
        )
        for document_id
        in ranked_documents[:k]
    ]

    ideal = sorted(
        graded_relevance.values(),
        reverse=True,
    )[:k]

    ideal_dcg = dcg(
        ideal
    )

    if ideal_dcg == 0:
        return 0.0

    return (
        dcg(gains)
        / ideal_dcg
    )

# 31. Retriever Comparison

In [ ]:
def ranked_documents_from_scores(
    scores,
    top_k,
):
    result = rank_frame(
        scores,
        top_k=top_k,
    )

    return result[
        "document_id"
    ].tolist()


def evaluate_method(
    method_name,
    retrieval_function,
    top_k=5,
):
    rows = []

    for example in benchmark:
        ranked_documents = (
            retrieval_function(
                example["query"],
                top_k,
            )
        )

        rows.append(
            {
                "method": method_name,
                "query": (
                    example["query"]
                ),
                "recall_at_k": (
                    recall_at_k(
                        ranked_documents,
                        example[
                            "relevant_documents"
                        ],
                        top_k,
                    )
                ),
                "precision_at_k": (
                    precision_at_k(
                        ranked_documents,
                        example[
                            "relevant_documents"
                        ],
                        top_k,
                    )
                ),
                "rr": (
                    reciprocal_rank(
                        ranked_documents,
                        example[
                            "relevant_documents"
                        ],
                    )
                ),
                "ap": (
                    average_precision(
                        ranked_documents,
                        example[
                            "relevant_documents"
                        ],
                    )
                ),
                "ndcg": (
                    ndcg(
                        ranked_documents,
                        example[
                            "graded_relevance"
                        ],
                        top_k,
                    )
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


def sparse_method(
    query,
    top_k,
):
    return ranked_documents_from_scores(
        sparse_scores(query),
        top_k,
    )


def dense_method(
    query,
    top_k,
):
    return ranked_documents_from_scores(
        dense_scores(query),
        top_k,
    )


def hybrid_method(
    query,
    top_k,
):
    return ranked_documents_from_scores(
        hybrid_scores(
            query,
            sparse_weight=0.55,
        ),
        top_k,
    )


def reranked_method(
    query,
    top_k,
):
    result = advanced_retrieve(
        query,
        candidate_k=max(
            8,
            top_k,
        ),
        top_k=top_k,
        sparse_weight=0.55,
        use_expansion=True,
    )

    return result[
        "document_id"
    ].tolist()


comparison_frames = [
    evaluate_method(
        "Sparse",
        sparse_method,
    ),
    evaluate_method(
        "Dense-SVD",
        dense_method,
    ),
    evaluate_method(
        "Hybrid",
        hybrid_method,
    ),
    evaluate_method(
        "Hybrid + rerank",
        reranked_method,
    ),
]

comparison_results = pd.concat(
    comparison_frames,
    ignore_index=True,
)

summary_metrics = (
    comparison_results.groupby(
        "method"
    )[
        [
            "recall_at_k",
            "precision_at_k",
            "rr",
            "ap",
            "ndcg",
        ]
    ]
    .mean()
    .sort_values(
        "ap",
        ascending=False,
    )
)

summary_metrics

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    summary_metrics.index,
    summary_metrics["ap"],
    marker="o",
    label="MAP",
)
plt.plot(
    summary_metrics.index,
    summary_metrics["ndcg"],
    marker="o",
    label="nDCG",
)
plt.xticks(rotation=25)
plt.ylabel("Score")
plt.title("Retriever Comparison")
plt.legend()
plt.tight_layout()
plt.show()

# 32. Chunk-Level Evaluation

Chunk-level evaluation is stricter than document-level evaluation because the
correct document can be retrieved while the wrong chunk is selected.

In [ ]:
chunk_relevance_example = pd.DataFrame(
    [
        (
            "doc_rag_metrics_chunk_00",
            "relevant",
        ),
        (
            "doc_rag_metrics_chunk_01",
            "partially relevant",
        ),
        (
            "doc_sparse_chunk_00",
            "not relevant",
        ),
    ],
    columns=[
        "chunk_id",
        "judgment",
    ],
)

chunk_relevance_example

# 33. Document-Level Evaluation

Document-level evaluation is useful when any passage from the correct source is
acceptable.

For production systems, report both levels when chunking quality matters.

# 34. Hard-Negative Analysis

In [ ]:
def hard_negatives(
    query,
    relevant_documents,
    top_k=6,
):
    retrieved = advanced_retrieve(
        query,
        candidate_k=10,
        top_k=top_k,
        sparse_weight=0.55,
        use_expansion=True,
    )

    return retrieved[
        ~retrieved[
            "document_id"
        ].isin(
            relevant_documents
        )
    ][
        [
            "rank",
            "rerank_score",
            "document_id",
            "title",
            "text",
        ]
    ]


first_example = benchmark[0]

hard_negatives(
    first_example["query"],
    first_example[
        "relevant_documents"
    ],
)

# 35. Context Budgeting

In [ ]:
def word_count(
    text: str,
) -> int:
    return len(
        text.split()
    )


def select_with_budget(
    ranked_frame: pd.DataFrame,
    word_budget: int,
) -> pd.DataFrame:
    selected_rows = []
    used_words = 0

    for row in ranked_frame.itertuples(
        index=False
    ):
        size = word_count(
            row.text
        )

        if (
            used_words
            + size
            > word_budget
        ):
            continue

        selected_rows.append(
            row._asdict()
        )

        used_words += size

    return pd.DataFrame(
        selected_rows
    )


ranked = advanced_retrieve(
    "How do retrieval metrics differ?",
    candidate_k=8,
    top_k=6,
)

select_with_budget(
    ranked,
    word_budget=55,
)

# 36. Diversity-Aware Context Selection

A context can waste space when several top chunks come from the same document.

A simple diversity rule keeps at most one chunk per document.

In [ ]:
def diverse_context(
    ranked_frame: pd.DataFrame,
    maximum_chunks: int = 3,
):
    selected = []
    seen_documents = set()

    for row in ranked_frame.itertuples(
        index=False
    ):
        if (
            row.document_id
            in seen_documents
        ):
            continue

        selected.append(
            row._asdict()
        )

        seen_documents.add(
            row.document_id
        )

        if (
            len(selected)
            >= maximum_chunks
        ):
            break

    return pd.DataFrame(
        selected
    )


diverse_context(
    advanced_retrieve(
        "Explain retrieval metrics and ranking quality.",
        candidate_k=10,
        top_k=8,
    ),
    maximum_chunks=3,
)

# 37. Retrieval Ablation

Ablation isolates the contribution of each component.

In [ ]:
ablation_rows = []

settings = [
    (
        "Sparse only",
        sparse_method,
    ),
    (
        "Dense only",
        dense_method,
    ),
    (
        "Hybrid",
        hybrid_method,
    ),
    (
        "Hybrid + expansion + rerank",
        reranked_method,
    ),
]

for name, function in settings:
    metrics = evaluate_method(
        name,
        function,
        top_k=5,
    )

    ablation_rows.append(
        {
            "setting": name,
            "MAP": metrics[
                "ap"
            ].mean(),
            "MRR": metrics[
                "rr"
            ].mean(),
            "nDCG": metrics[
                "ndcg"
            ].mean(),
        }
    )

pd.DataFrame(
    ablation_rows
)

# 38. Failure Taxonomy

In [ ]:
failure_taxonomy = pd.DataFrame(
    [
        ("Embedding failure", "semantic match not represented"),
        ("Lexical failure", "rare exact term underweighted"),
        ("Filter failure", "correct evidence removed by metadata"),
        ("Expansion drift", "added terms change query intent"),
        ("Candidate failure", "relevant passage never reaches reranker"),
        ("Reranker failure", "hard negative ranked above relevant passage"),
        ("Context failure", "useful evidence dropped by budget"),
    ],
    columns=[
        "Failure",
        "Description",
    ],
)

failure_taxonomy

# 39. Dense Embedding Model Considerations

When replacing SVD with a neural embedding model, evaluate:

- embedding dimension;
- similarity function;
- domain fit;
- multilingual support;
- maximum input length;
- normalization;
- batch size;
- hardware cost.

# 40. Vector Database Considerations

Important system-level choices include:

- exact versus approximate nearest-neighbor search;
- index update frequency;
- metadata filtering;
- persistence;
- sharding;
- tenant isolation;
- access control;
- deletion semantics.

# 41. Multilingual Retrieval

Multilingual dense retrieval should test:

- same-language retrieval;
- cross-language retrieval;
- code-switching;
- translated queries;
- language imbalance;
- embedding-space alignment.

# 42. Arabic Dense Retrieval

Arabic dense retrieval must account for:

- rich morphology;
- attached clitics;
- optional tashkeel;
- orthographic variants;
- MSA and dialects;
- cross-lingual embedding quality.

In [ ]:
arabic_cases = pd.DataFrame(
    [
        (
            "وَسَيَكْتُبُونَهَا",
            "fully vocalized complex form",
            "preserve tashkeel in vocalized tasks",
        ),
        (
            "بِالْمَدْرَسَةِ",
            "clitic-rich token",
            "test lexical and dense matching",
        ),
        (
            "كِتَابُهُمَا",
            "stem plus dual pronoun",
            "evaluate morphology-sensitive retrieval",
        ),
    ],
    columns=[
        "Form",
        "Property",
        "Evaluation concern",
    ],
)

arabic_cases

For fully vocalized Arabic retrieval, the query, stored chunk, embedding input, and
evaluation labels should retain tashkeel consistently.

# 43. Optional Sentence-Transformer Workflow

This optional template is disabled by default.

In [ ]:
SENTENCE_TRANSFORMERS_AVAILABLE = (
    importlib.util.find_spec(
        "sentence_transformers"
    )
    is not None
)

RUN_NEURAL_EMBEDDING_DEMO = False

neural_embedding_template = '''
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

passage_embeddings = model.encode(
    chunk_frame["text"].tolist(),
    normalize_embeddings=True,
)

query_embedding = model.encode(
    [query],
    normalize_embeddings=True,
)

scores = query_embedding @ passage_embeddings.T
'''

print(
    "sentence-transformers installed:",
    SENTENCE_TRANSFORMERS_AVAILABLE,
)
print(
    "Optional neural demo enabled:",
    RUN_NEURAL_EMBEDDING_DEMO,
)
print(neural_embedding_template)

# 44. Reproducibility

Record:

- corpus version;
- chunking configuration;
- sparse vectorizer;
- dense embedding model or projection;
- normalization;
- similarity function;
- hybrid fusion method;
- query expansion rules;
- metadata filters;
- candidate count;
- reranker;
- final top-k;
- context budget;
- benchmark judgments;
- MAP, MRR, nDCG, recall@k, and precision@k.

In [ ]:
reproducibility_record = pd.Series(
    {
        "module": (
            "Module 8 • Large Language Models"
        ),
        "lesson": (
            "Lesson 46 • Advanced RAG — Dense Embeddings, "
            "Re-Ranking, and Retrieval Evaluation"
        ),
        "documents": len(
            documents
        ),
        "chunks": len(
            chunk_frame
        ),
        "sparse representation": (
            "TF-IDF unigram + bigram"
        ),
        "dense representation": (
            f"TruncatedSVD({maximum_components})"
        ),
        "hybrid sparse weight": 0.55,
        "candidate_k": 8,
        "benchmark queries": len(
            benchmark
        ),
        "offline execution": True,
        "python": (
            platform.python_version()
        ),
    },
    name="Lesson 46 experiment",
)

reproducibility_record

# 45. Knowledge Check

1. How do sparse and dense retrieval differ?
2. Why can dense retrieval help with paraphrases?
3. Why can sparse retrieval remain important?
4. What is hybrid retrieval?
5. What does the hybrid mixing weight control?
6. Why use metadata filters?
7. What is query expansion?
8. What is a hard negative?
9. Why use a two-stage retriever?
10. How does reranking differ from first-stage retrieval?
11. What does MAP measure?
12. What does nDCG add beyond binary relevance metrics?
13. Why report chunk-level and document-level results separately?
14. Why enforce a context budget?
15. Why test Arabic lexical and dense matching separately?

# 46. Exercises

## Exercise 1 — Hybrid Weight Sweep
Sweep sparse weights from 0.0 to 1.0.

## Exercise 2 — Query Expansion
Add acronym and synonym expansion.

## Exercise 3 — Metadata Filters
Add date and access-group filtering.

## Exercise 4 — Re-Ranking
Replace the heuristic reranker with a learned model.

## Exercise 5 — Hard Negatives
Build a hard-negative set for each benchmark query.

## Exercise 6 — MAP and nDCG
Compare methods under graded relevance.

## Exercise 7 — Context Budget
Optimize answer evidence under a fixed word budget.

## Exercise 8 — Neural Embeddings
Replace SVD with a sentence-transformer model.

## Exercise 9 — Arabic Dense Retrieval
Build a fully vocalized Arabic retrieval benchmark.

## Exercise 10 — Retrieval Report
Document all retrieval stages and ablation results.

## Challenge Exercises

1. Add reciprocal-rank fusion.
2. Implement maximal marginal relevance.
3. Add a cross-encoder reranker.
4. Compare approximate nearest-neighbor indexes.
5. Build a multilingual English–Arabic hybrid retrieval benchmark.

# 47. Summary and Next Lesson

In this lesson:

- sparse and dense retrieval were compared;
- TF-IDF plus truncated SVD provided an offline dense-vector approximation;
- hybrid retrieval combined lexical and semantic relevance;
- metadata filtering and query expansion were added;
- two-stage retrieval and reranking were implemented;
- recall@k, precision@k, MRR, MAP, and nDCG were calculated;
- document-level and chunk-level evaluation were distinguished;
- hard negatives were inspected;
- context budgeting and diversity-aware selection were implemented;
- retrieval ablations quantified component contributions;
- vector-database, multilingual, Arabic, and tashkeel considerations were covered.

## Next Lesson

**Lesson 47: LLM Evaluation, Hallucination, and Reliability** covers capability
evaluation, groundedness, factuality, calibration, robustness, prompt sensitivity,
citation evaluation, human evaluation, and systematic reliability testing.

# References

- Karpukhin, V. et al. *Dense Passage Retrieval for Open-Domain Question Answering*.
- Nogueira, R., & Cho, K. *Passage Re-ranking with BERT*.
- Robertson, S., & Zaragoza, H. *The Probabilistic Relevance Framework: BM25 and Beyond*.
- Järvelin, K., & Kekäläinen, J. *Cumulated Gain-Based Evaluation of IR Techniques*.
- Lewis, P. et al. *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*.